## Setup — Bibliotecas, Parâmetros e Funções Utilitárias

Antes de iniciar as transformações da camada Silver, este bloco prepara o ambiente com três tipos de recurso:

**1. Imports**
- `pyspark.sql.functions as F`, `Row` e `Window`: funções nativas do Spark para transformação de colunas, criação de registros estruturados (usados no log de DQ) e operações analíticas (ex.: deduplicação por partição).
- `datetime`: usado no parsing multi-formato de datas e no timestamp de cada checagem de qualidade.
- `StructType`/`StructField`/`StringType`/`DoubleType`/`IntegerType`: tipos para definição explícita de schema, evitando inferência automática incorreta em colunas problemáticas da origem.
- `re`: suporte a limpeza de strings via expressões regulares (ex.: remoção de símbolos monetários e ruídos de formatação).

**2. Parâmetros de Ambiente**
Centralizam os nomes de catálogo e schemas (`bronze`, `silver`, `gold`) em variáveis reutilizáveis, evitando strings hardcoded espalhadas pelo notebook e facilitando a portabilidade entre ambientes (dev/prod).

**3. Framework de Qualidade de Dados (`dq_check` e `dq_check_unique`)**
Funções utilitárias para validar as regras de negócio aplicadas em cada tabela Silver, registrando o resultado (linhas totais, linhas que falharam, status PASS/FAIL) na lista `dq_results` para consolidação e auditoria posterior.
- `dq_check`: valida uma condição booleana genérica (ex.: valores dentro de um intervalo válido).
- `dq_check_unique`: valida especificamente a unicidade de uma ou mais colunas-chave, identificando duplicidades na granularidade esperada da tabela.

> **Observação:** essas funções são checagens **informativas** — elas contam e reportam violações no console, mas não interrompem a gravação da tabela nem filtram automaticamente as linhas problemáticas.

**4. `parse_date_robust`**
Função auxiliar para tratar o requisito de "Tratamento de Data Multi-Formato" da Silver: testa sequencialmente múltiplos padrões de data (`YYYY-MM-DD`, `DD/MM/YYYY`, `MM/DD/YYYY`, etc.) presentes na origem e retorna `None` apenas quando nenhum formato é compatível, evitando descartar registros por um parsing rígido demais.

In [0]:
from pyspark.sql import functions as F, Row, Window
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import re

# Parâmetros do ambiente (reutilizados da Bronze)
catalog = "cinedata"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Lista para armazenar resultados de qualidade de dados
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")


def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total}")


def parse_date_robust(date_str):
    """Tenta converter string de data em diferentes formatos. Retorna NULL se falhar."""
    if not date_str or date_str.strip() == "":
        return None
    
    date_str = str(date_str).strip()
    formats = [
        "%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%d-%m-%Y",
        "%Y/%m/%d", "%d.%m.%Y", "%Y.%m.%d"
    ]
    
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt).strftime("%Y-%m-%d")
        except:
            continue
    return None

print("Setup completo: Bibliotecas importadas, funções de qualidade definidas")


Setup completo: Bibliotecas importadas, funções de qualidade definidas


---
## BLOCO 1: silver.tb_info_filmes

**Origem:** bronze.tb_movies_info

### Regras de Negócio Aplicadas:

#### REGRA 1: Renomear colunas para português
`Motivo: Padronização do projeto (todas as colunas Silver devem estar em português)`

#### REGRA 2: Limpeza e tradução do status_filme
`Motivo: Status vem em inglês da fonte; padronização exige português, remoção de ruídos (espaços extra, case inconsistente) e mapeamento dos status reconhecidos pela indústria`

#### REGRA 3: Normalizar nomes (remover espaços extra, capitalizar)
`Motivo: Dados podem vir com espaçamento incorreto ou case inconsistente`

#### REGRA 4: Converter data_lancamento com múltiplos formatos
`Motivo: Dados de diferentes fontes (TMDB, IMDB) podem vir em formatos diferentes (YYYY-MM-DD vs DD/MM/YYYY). Função UDF testa múltiplos padrões`

#### REGRA 5: Criar coluna derivada: ano_lancamento
`Motivo: Análises frequentes agrupam por ano; extrair no Silver economiza reprocessamento na Gold`

#### REGRA 6: Deduplicação por id_filme
`Motivo: IMDB e TMDB podem ter registros duplicados com dados ligeiramente diferentes. Manter sempre o mais recente garante informações atualizadas`

#### REGRA 7: Filtrar nulos estruturais
`Motivo: id_filme é chave primária; registros sem ID são inutilizáveis`

***

In [0]:
print("\n[1/7] Transformando: silver.tb_info_filmes\n")

df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

df_silver_info = (
    df_bronze_info
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
)

df_silver_info = df_silver_info.withColumn(
    "status_filme",
    F.lower(F.trim(F.col("status_filme")))
)

df_silver_info = df_silver_info.withColumn(
    "status_filme",
    F.when(F.col("status_filme") == "released", "Lançado")
    .when(F.col("status_filme") == "post production", "Pós-Produção")
    .when(F.col("status_filme") == "in production", "Em Produção")
    .when(F.col("status_filme") == "planned", "Planejado")
    .when(F.col("status_filme") == "rumored", "Boato")
    .when(F.col("status_filme") == "canceled", "Cancelado")
    .otherwise("Desconhecido")
)

df_silver_info = (
    df_silver_info
    .withColumn("titulo", F.initcap(F.trim(F.col("titulo"))))
    .withColumn("titulo_original", F.initcap(F.trim(F.col("titulo_original"))))
    .withColumn("idioma_original", F.upper(F.trim(F.col("idioma_original"))))
)

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType as ST

parse_date_udf = udf(parse_date_robust, ST())

df_silver_info = df_silver_info.withColumn(
    "data_lancamento",
    F.to_date(parse_date_udf(F.col("data_lancamento")), "yyyy-MM-dd")
)

df_silver_info = df_silver_info.withColumn(
    "ano_lancamento",
    F.year(F.col("data_lancamento"))
)

df_silver_info = (
    df_silver_info
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("id_filme").orderBy(F.desc("ingestion_datetime"))
    ))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_silver_info = df_silver_info.filter(F.col("id_filme").isNotNull())

df_silver_info = df_silver_info.select(
    "id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao"
)

print("Quality Checks para silver.tb_info_filmes:\n")
dq_check_unique("silver.tb_info_filmes", "id_filme único", df_silver_info, ["id_filme"])
dq_check("silver.tb_info_filmes", "id_filme não nulo", df_silver_info, F.col("id_filme").isNotNull())
dq_check("silver.tb_info_filmes", "data_lancamento não nula", df_silver_info, F.col("data_lancamento").isNotNull())
dq_check("silver.tb_info_filmes", "status_filme em domínio válido", df_silver_info,
         F.col("status_filme").isin(["Lançado", "Pós-Produção", "Em Produção", "Planejado", "Boato", "Cancelado", "Desconhecido"]))

df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_info_filmes")

print(f"\n✓ silver.tb_info_filmes gravada com {df_silver_info.count()} registros únicos")



[1/7] Transformando: silver.tb_info_filmes

Quality Checks para silver.tb_info_filmes:

[PASS] silver.tb_info_filmes | id_filme único | 0 chaves duplicadas de 97879
[PASS] silver.tb_info_filmes | id_filme não nulo | 0/97879 linhas falharam
[FAIL] silver.tb_info_filmes | data_lancamento não nula | 2565/97879 linhas falharam
[PASS] silver.tb_info_filmes | status_filme em domínio válido | 0/97879 linhas falharam

✓ silver.tb_info_filmes gravada com 97879 registros únicos


---

## BLOCO 2: silver.tb_financeiro_filmes

**Origem:** bronze.tb_movies_financials

### Regras de Negócio Aplicadas:

#### REGRA 1: Renomear colunas para português
`Motivo: Padronização multilíngue do projeto`

#### REGRA 2: Higienizar e converter para DECIMAL
`Motivo: Valores monetários vêm com símbolos ($, €, espaços, pontos de milhar). Necessário remover TUDO exceto dígitos e ponto decimal ANTES de converter para DECIMAL(18,2)`

#### REGRA 3: Remover valores zerados ou negativos
`Motivo: Orçamento e receita negativos ou zero não fazem sentido financeiro. São indicadores de erro ou dados faltantes mascarados`

#### REGRA 4: Calcular equivalentes em BRL
`Motivo: Análises do negócio necessitam valores em BRL (moeda local). Usar taxa de cotação obtida da API`

#### REGRA 5: Derivar Lucro (USD e BRL)
`Motivo: Métrica crítica para análise de performance. Lucro = Receita - Orçamento. Se qualquer um for nulo, resultado é nulo`

#### REGRA 6: Derivar Margem de Lucro Percentual
`Motivo: Indicador financeiro importante (% de retorno sobre investimento). Proteção contra divisão por zero`

***

In [0]:
print("\n[2/7] Transformando: silver.tb_financeiro_filmes\n")

df_bronze_financials = spark.table(f"{bronze_schema}.tb_movies_financials")
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

taxa_cotacao = df_bronze_cotacao.orderBy(F.desc("data_cotacao")).limit(1).select(F.col("cotacao_usd_brl")).collect()[0][0]
print(f"Taxa de cotação USD/BRL utilizada: {taxa_cotacao:.4f}")

df_silver_financials = (
    df_bronze_financials
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")
)

df_silver_financials = (
    df_silver_financials
    .withColumn("orcamento_usd",
                F.when(F.col("orcamento_usd").isin(["Unknown", "Não Informado", "NA", "N/A", "", "null"]), None)
                .otherwise(F.regexp_replace(F.col("orcamento_usd"), r"[^0-9.]", "").cast("decimal(18,2)")))
    .withColumn("receita_usd",
                F.when(F.col("receita_usd").isin(["Unknown", "Não Informado", "NA", "N/A", "", "null"]), None)
                .otherwise(F.regexp_replace(F.col("receita_usd"), r"[^0-9.]", "").cast("decimal(18,2)")))
)

df_silver_financials = (
    df_silver_financials
    .withColumn("orcamento_usd", F.when(F.col("orcamento_usd") <= 0, None).otherwise(F.col("orcamento_usd")))
    .withColumn("receita_usd", F.when(F.col("receita_usd") <= 0, None).otherwise(F.col("receita_usd")))
)

df_silver_financials = (
    df_silver_financials
    .withColumn("orcamento_brl", F.when(F.col("orcamento_usd").isNotNull(), F.round(F.col("orcamento_usd") * F.lit(taxa_cotacao), 2)).otherwise(None))
    .withColumn("receita_brl", F.when(F.col("receita_usd").isNotNull(), F.round(F.col("receita_usd") * F.lit(taxa_cotacao), 2)).otherwise(None))
)

df_silver_financials = (
    df_silver_financials
    .withColumn("lucro_usd", F.when((F.col("receita_usd").isNotNull()) & (F.col("orcamento_usd").isNotNull()), F.round(F.col("receita_usd") - F.col("orcamento_usd"), 2)).otherwise(None))
    .withColumn("lucro_brl", F.when((F.col("receita_brl").isNotNull()) & (F.col("orcamento_brl").isNotNull()), F.round(F.col("receita_brl") - F.col("orcamento_brl"), 2)).otherwise(None))
)

df_silver_financials = (
    df_silver_financials
    .withColumn("margem_lucro_pct", F.when((F.col("receita_usd").isNotNull()) & (F.col("receita_usd") != 0), F.round((F.col("lucro_usd") / F.col("receita_usd")) * 100, 2)).otherwise(None))
)

df_silver_financials = df_silver_financials.select(
    "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl", "lucro_usd", "lucro_brl", "margem_lucro_pct"
)

print("Quality Checks para silver.tb_financeiro_filmes:\n")
dq_check("silver.tb_financeiro_filmes", "orcamento_usd >= 0", df_silver_financials, (F.col("orcamento_usd").isNull()) | (F.col("orcamento_usd") > 0))
dq_check("silver.tb_financeiro_filmes", "receita_usd >= 0", df_silver_financials, (F.col("receita_usd").isNull()) | (F.col("receita_usd") > 0))
dq_check("silver.tb_financeiro_filmes", "margem_lucro_pct entre -100 e 100", df_silver_financials, (F.col("margem_lucro_pct").isNull()) | ((F.col("margem_lucro_pct") >= -100) & (F.col("margem_lucro_pct") <= 1000)))

df_silver_financials.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

print(f"\n✓ silver.tb_financeiro_filmes gravada com {df_silver_financials.count()} registros")



[2/7] Transformando: silver.tb_financeiro_filmes

Taxa de cotação USD/BRL utilizada: 5.1111
Quality Checks para silver.tb_financeiro_filmes:

[PASS] silver.tb_financeiro_filmes | orcamento_usd >= 0 | 0/743155 linhas falharam
[PASS] silver.tb_financeiro_filmes | receita_usd >= 0 | 0/743155 linhas falharam
[FAIL] silver.tb_financeiro_filmes | margem_lucro_pct entre -100 e 100 | 2744/743155 linhas falharam

✓ silver.tb_financeiro_filmes gravada com 743155 registros


---

## BLOCO 3: silver.tb_metricas_engajamento

**Origem:** bronze.tb_movies_metrics

### Regras de Negócio Aplicadas:

#### REGRA 1: Renomear colunas para português
`Motivo: Padronização multilíngue do projeto`

#### REGRA 2: Higienizar e converter para tipos corretos
`Motivo: Dados vêm com notação europeia de milhar (7.474.8887 = 7474.8887). Usar UDF Python para remover caracteres não numéricos e pontos múltiplos corretamente`

#### REGRA 3: Validar notas (intervalo 0-10)
`Motivo: Escalas de nota definem domínio [0,10]. Valores fora desse intervalo são erros ou dados malformados. Tratar como NULL evita enviesamento de análises`

#### REGRA 4: Popularidade deve ser sempre positiva
`Motivo: Índice de popularidade é derivado de atividade; valor negativo indica erro ou dado fora do domínio esperado`

#### REGRA 5: Contagens de votos devem ser não-negativas
`Motivo: Não faz sentido ter -5 votos; é um erro de entrada`


In [0]:
print("\n[3/7] Transformando: silver.tb_metricas_engajamento\n")

df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

df_silver_metrics = (
    df_bronze_metrics
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
)

def safe_numeric_convert(val, is_int=False):
    try:
        if not val or str(val).strip() == "":
            return None
        val_str = str(val).strip()
        cleaned = re.sub(r"[^0-9.,]", "", val_str)
        if not cleaned or cleaned == "":
            return None
        cleaned = cleaned.replace(",", ".")
        parts = cleaned.split(".")
        if len(parts) > 2:
            cleaned = "".join(parts[:-1]) + "." + parts[-1]
        if cleaned.endswith("."):
            cleaned = cleaned[:-1]
        if not cleaned or cleaned == "":
            return None
        if is_int:
            return int(float(cleaned))
        else:
            return float(cleaned)
    except:
        return None

safe_double_udf = F.udf(lambda x: safe_numeric_convert(x, is_int=False), DoubleType())
safe_int_udf = F.udf(lambda x: safe_numeric_convert(x, is_int=True), IntegerType())

df_silver_metrics = (
    df_silver_metrics
    .withColumn("popularidade", safe_double_udf(F.col("popularidade")))
    .withColumn("nota_media_tmdb", safe_double_udf(F.col("nota_media_tmdb")))
    .withColumn("qtd_votos_tmdb", safe_int_udf(F.col("qtd_votos_tmdb")))
    .withColumn("nota_media_imdb", safe_double_udf(F.col("nota_media_imdb")))
    .withColumn("qtd_votos_imdb", safe_int_udf(F.col("qtd_votos_imdb")))
)

df_silver_metrics = (
    df_silver_metrics
    .withColumn("nota_media_tmdb", F.when((F.col("nota_media_tmdb") >= 0) & (F.col("nota_media_tmdb") <= 10), F.col("nota_media_tmdb")).otherwise(None))
    .withColumn("nota_media_imdb", F.when((F.col("nota_media_imdb") >= 0) & (F.col("nota_media_imdb") <= 10), F.col("nota_media_imdb")).otherwise(None))
)

df_silver_metrics = (
    df_silver_metrics
    .withColumn("popularidade", F.when(F.col("popularidade") < 0, None).otherwise(F.col("popularidade")))
)

df_silver_metrics = (
    df_silver_metrics
    .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") < 0, None).otherwise(F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") < 0, None).otherwise(F.col("qtd_votos_imdb")))
)

df_silver_metrics = df_silver_metrics.select(
    "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"
)

print("Quality Checks para silver.tb_metricas_engajamento:\n")
dq_check("silver.tb_metricas_engajamento", "nota_media_tmdb entre 0 e 10", df_silver_metrics, (F.col("nota_media_tmdb").isNull()) | ((F.col("nota_media_tmdb") >= 0) & (F.col("nota_media_tmdb") <= 10)))
dq_check("silver.tb_metricas_engajamento", "nota_media_imdb entre 0 e 10", df_silver_metrics, (F.col("nota_media_imdb").isNull()) | ((F.col("nota_media_imdb") >= 0) & (F.col("nota_media_imdb") <= 10)))
dq_check("silver.tb_metricas_engajamento", "popularidade >= 0", df_silver_metrics, (F.col("popularidade").isNull()) | (F.col("popularidade") >= 0))

df_silver_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

print(f"\n✓ silver.tb_metricas_engajamento gravada com {df_silver_metrics.count()} registros")



[3/7] Transformando: silver.tb_metricas_engajamento

Quality Checks para silver.tb_metricas_engajamento:

[PASS] silver.tb_metricas_engajamento | nota_media_tmdb entre 0 e 10 | 0/751548 linhas falharam
[PASS] silver.tb_metricas_engajamento | nota_media_imdb entre 0 e 10 | 0/751548 linhas falharam
[PASS] silver.tb_metricas_engajamento | popularidade >= 0 | 0/751548 linhas falharam

✓ silver.tb_metricas_engajamento gravada com 751548 registros


---

## BLOCO 4: silver.tb_avaliacoes_usuarios

**Origem:** bronze.tb_movies_reviews

### Regras de Negócio Aplicadas:

#### REGRA 1: Renomear colunas para português
`Motivo: Padronização multilíngue do projeto`

#### REGRA 2: Converter nota para tipo correto
`Motivo: Escala de avaliação é 0-10. Valores fora desse intervalo indicam erro de entrada`

#### REGRA 3: Filtrar notas fora do intervalo
`Motivo: Descartar valores inválidos que distorceriam análises`

#### REGRA 4: Tratar comentários vazios
`Motivo: Normalizar representação de "falta de comentário". Usuários votam mas não deixam comentário; representar consistentemente como "Sem comentário"`

#### REGRA 5: Remover duplicatas inteiras
`Motivo: Usuários podem avaliar mesmo filme múltiplas vezes. Manter apenas ocorrência mais recente`

***

In [0]:
print("\n[4/7] Transformando: silver.tb_avaliacoes_usuarios\n")

df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

df_silver_reviews = (
    df_bronze_reviews
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

df_silver_reviews = df_silver_reviews.withColumn("nota_usuario", F.col("nota_usuario").cast("double"))

df_silver_reviews = df_silver_reviews.filter((F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10))

df_silver_reviews = df_silver_reviews.withColumn(
    "comentario_usuario",
    F.when((F.col("comentario_usuario").isNull()) | (F.trim(F.col("comentario_usuario")) == ""), "Sem comentário")
    .otherwise(F.trim(F.col("comentario_usuario")))
)

df_silver_reviews = (
    df_silver_reviews
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario").orderBy(F.desc("ingestion_datetime"))
    ))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_silver_reviews = df_silver_reviews.select("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario")

print("Quality Checks para silver.tb_avaliacoes_usuarios:\n")
dq_check("silver.tb_avaliacoes_usuarios", "nota_usuario entre 0 e 10", df_silver_reviews, (F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10))
dq_check("silver.tb_avaliacoes_usuarios", "nome_usuario não nulo", df_silver_reviews, F.col("nome_usuario").isNotNull())
dq_check("silver.tb_avaliacoes_usuarios", "comentario_usuario nunca vazio", df_silver_reviews, F.col("comentario_usuario").isNotNull())

df_silver_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

print(f"\n✓ silver.tb_avaliacoes_usuarios gravada com {df_silver_reviews.count()} registros únicos")



[4/7] Transformando: silver.tb_avaliacoes_usuarios

Quality Checks para silver.tb_avaliacoes_usuarios:

[PASS] silver.tb_avaliacoes_usuarios | nota_usuario entre 0 e 10 | 0/30761 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | nome_usuario não nulo | 0/30761 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | comentario_usuario nunca vazio | 0/30761 linhas falharam

✓ silver.tb_avaliacoes_usuarios gravada com 30761 registros únicos


---

## BLOCO 5: silver.tb_generos

**Origem:** bronze.tb_credits_and_tags (coluna: genres)

### Regras de Negócio Aplicadas:

#### REGRA 1: Selecionar apenas colunas necessárias
`Motivo: Reduzir dados para operação que precisa apenas de id e genres`

#### REGRA 2: Explodir coluna de gêneros
`Motivo: Dados vêm como lista delimitada ("Action|Drama|Thriller"). Necessário transformar em uma linha por gênero para dimensão única`

#### REGRA 3: Normalizar gêneros
`Motivo: Dados sujos podem ter espaços extras, case inconsistente. Remove valores como "null", "", espaços em branco`

#### REGRA 4: Capitalizar nomes de gêneros
`Motivo: Padronização (ex: "Action" não "ACTION", "action", "AcTiOn")`

#### REGRA 5: Deduplicar gêneros por filme
`Motivo: Casos onde o mesmo gênero aparece múltiplas vezes para o mesmo filme. Manter apenas uma ocorrência`

***

In [0]:
print("\n[5/7] Transformando: silver.tb_generos\n")

df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_genres = df_bronze_credits.select(F.col("id").alias("id_filme"), F.col("genres"))

# Normaliza separadores: vírgula e ponto-e-vírgula ambos viram o mesmo delimitador antes do split
df_genres = df_genres.withColumn(
    "genres_normalizado",
    F.regexp_replace(F.col("genres"), "[;,]", "|")
)

df_genres = (
    df_genres
    .withColumn("genero", F.explode(F.split(F.col("genres_normalizado"), "\\|")))
    .drop("genres", "genres_normalizado")
)

df_genres = df_genres.withColumn("genero", F.trim(F.col("genero")))
df_genres = df_genres.withColumn("genero", F.initcap(F.col("genero")))

# Whitelist: domínio fechado de gêneros válidos (TMDB), elimina resíduos de Column Shift
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
    "Romance", "Science Fiction", "Tv Movie", "Thriller", "War", "Western"
]

df_genres = df_genres.filter(
    (F.col("genero").isNotNull()) &
    (F.col("genero") != "") &
    (F.col("genero").isin(generos_validos))
)

df_genres = df_genres.distinct()

print("Quality Checks para silver.tb_generos:\n")
dq_check("silver.tb_generos", "id_filme não nulo", df_genres, F.col("id_filme").isNotNull())
dq_check("silver.tb_generos", "genero não vazio", df_genres, (F.col("genero").isNotNull()) & (F.col("genero") != ""))
dq_check("silver.tb_generos", "genero no domínio válido", df_genres, F.col("genero").isin(generos_validos))

df_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_generos")

print(f"\n✓ silver.tb_generos gravada com {df_genres.count()} registros")


[5/7] Transformando: silver.tb_generos

Quality Checks para silver.tb_generos:

[PASS] silver.tb_generos | id_filme não nulo | 0/141965 linhas falharam
[PASS] silver.tb_generos | genero não vazio | 0/141965 linhas falharam
[PASS] silver.tb_generos | genero no domínio válido | 0/141965 linhas falharam

✓ silver.tb_generos gravada com 141965 registros


---

## BLOCO 6: silver.tb_pessoas_empresas

**Origem:** bronze.tb_credits_and_tags (colunas: cast, directors, writers, production_companies)

### Regras de Negócio Aplicadas:

#### REGRA 1: Preparar cada tipo de entidade separadamente
`Motivo: Cada coluna tem estrutura diferente; processaremos e consolidaremos em única tabela com tipo_entidade`

#### REGRA 2: Consolidar 4 tabelas em uma única
`Motivo: Dimensão unificada permite análises cruzadas (ex: contar filmes por ator/diretor/etc)`

#### REGRA 3: Normalizar nomes
`Motivo: Dados sujos podem ter espaços extras, case inconsistente. Capitalização padrão melhora legibilidade e evita duplicação`

#### REGRA 4: Remover entradas vazias
`Motivo: Linhas em branco não têm significado; descartar economiza espaço`

#### REGRA 5: Eliminar duplicatas
`Motivo: Mesma pessoa/empresa pode aparecer múltiplas vezes para o mesmo filme. Manter apenas uma ocorrência`

***

In [0]:
print("\n[6/7] Transformando: silver.tb_pessoas_empresas\n")

df_bronze_credits_full = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_cast = (
    df_bronze_credits_full
    .select(F.col("id").alias("id_filme"), F.col("cast"))
    .withColumn("nome_entidade", F.explode(F.split(F.col("cast"), "\\|")))
    .withColumn("tipo_entidade", F.lit("Ator"))
    .drop("cast")
)

df_directors = (
    df_bronze_credits_full
    .select(F.col("id").alias("id_filme"), F.col("directors"))
    .withColumn("nome_entidade", F.explode(F.split(F.col("directors"), "\\|")))
    .withColumn("tipo_entidade", F.lit("Diretor"))
    .drop("directors")
)

df_writers = (
    df_bronze_credits_full
    .select(F.col("id").alias("id_filme"), F.col("writers"))
    .withColumn("nome_entidade", F.explode(F.split(F.col("writers"), "\\|")))
    .withColumn("tipo_entidade", F.lit("Roteirista"))
    .drop("writers")
)

df_companies = (
    df_bronze_credits_full
    .select(F.col("id").alias("id_filme"), F.col("production_companies"))
    .withColumn("nome_entidade", F.explode(F.split(F.col("production_companies"), "\\|")))
    .withColumn("tipo_entidade", F.lit("Produtora"))
    .drop("production_companies")
)

df_pessoas_empresas = df_cast.union(df_directors).union(df_writers).union(df_companies)

df_pessoas_empresas = (
    df_pessoas_empresas
    .withColumn("nome_entidade", F.trim(F.col("nome_entidade")))
    .withColumn("nome_entidade", F.initcap(F.col("nome_entidade")))
)

df_pessoas_empresas = df_pessoas_empresas.filter(
    (F.col("nome_entidade").isNotNull()) & (F.col("nome_entidade") != "")
)

df_pessoas_empresas = df_pessoas_empresas.distinct()

df_pessoas_empresas = df_pessoas_empresas.select("id_filme", "nome_entidade", "tipo_entidade")

print("Quality Checks para silver.tb_pessoas_empresas:\n")
dq_check("silver.tb_pessoas_empresas", "id_filme não nulo", df_pessoas_empresas, F.col("id_filme").isNotNull())
dq_check("silver.tb_pessoas_empresas", "nome_entidade não vazio", df_pessoas_empresas, (F.col("nome_entidade").isNotNull()) & (F.col("nome_entidade") != ""))
dq_check("silver.tb_pessoas_empresas", "tipo_entidade em domínio válido", df_pessoas_empresas, F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista", "Produtora"]))

df_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

print(f"\n✓ silver.tb_pessoas_empresas gravada com {df_pessoas_empresas.count()} registros únicos")



[6/7] Transformando: silver.tb_pessoas_empresas

Quality Checks para silver.tb_pessoas_empresas:

[PASS] silver.tb_pessoas_empresas | id_filme não nulo | 0/328589 linhas falharam
[PASS] silver.tb_pessoas_empresas | nome_entidade não vazio | 0/328589 linhas falharam
[PASS] silver.tb_pessoas_empresas | tipo_entidade em domínio válido | 0/328589 linhas falharam

✓ silver.tb_pessoas_empresas gravada com 328589 registros únicos


---

## BLOCO 7: silver.tb_cotacao_dolar

**Origem:** bronze.tb_cotacao_dolar

### Regras de Negócio Aplicadas:

#### REGRA 1: Converter data_cotacao para TIMESTAMP
`Motivo: Converter para TIMESTAMP permite operações temporais precisas`

#### REGRA 2: Ordenar por data (série temporal)
`Motivo: Forward Fill e análises temporais exigem dados em sequência cronológica`

#### REGRA 3: Aplicar Forward Fill
`Motivo: Banco Central não publica cotações em fins de semana e feriados. Para esses dias, usar última cotação disponível garante conversões USD→BRL sem gaps temporais`

***

In [0]:
print("\n[7/7] Transformando: silver.tb_cotacao_dolar\n")

df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_silver_cotacao = df_bronze_cotacao.withColumn(
    "data_cotacao",
    F.to_timestamp(F.col("data_cotacao"))
)

df_silver_cotacao = df_silver_cotacao.sort(F.col("data_cotacao"))

df_silver_cotacao = (
    df_silver_cotacao
    .withColumn("cotacao_preenchida", 
                F.last(F.col("cotacao_usd_brl"), ignorenulls=True)
                .over(Window.orderBy(F.col("data_cotacao"))))
    .withColumn("cotacao_usd_brl", F.col("cotacao_preenchida"))
    .drop("cotacao_preenchida")
)

df_silver_cotacao = df_silver_cotacao.select("data_cotacao", "cotacao_usd_brl")

print("Quality Checks para silver.tb_cotacao_dolar:\n")
dq_check("silver.tb_cotacao_dolar", "cotacao_usd_brl > 0", df_silver_cotacao, F.col("cotacao_usd_brl") > 0)
dq_check("silver.tb_cotacao_dolar", "data_cotacao não nula", df_silver_cotacao, F.col("data_cotacao").isNotNull())

df_silver_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

print(f"\n✓ silver.tb_cotacao_dolar gravada com {df_silver_cotacao.count()} registros")



[7/7] Transformando: silver.tb_cotacao_dolar

Quality Checks para silver.tb_cotacao_dolar:



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] silver.tb_cotacao_dolar | cotacao_usd_brl > 0 | 0/32 linhas falharam
[PASS] silver.tb_cotacao_dolar | data_cotacao não nula | 0/32 linhas falharam

✓ silver.tb_cotacao_dolar gravada com 32 registros


---

## BLOCO 8: Validação Final

In [0]:
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("✓ VALIDAÇÃO FINAL — CAMADA SILVER")
print("=" * 80 + "\n")

tables = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

all_ok = True

print("Contagem de registros por tabela:\n")
for table_name in tables:
    full_table_name = f"{silver_schema}.{table_name}"
    try:
        count = spark.table(full_table_name).count()
        print(f"  ✓ {table_name}: {count:,} registros")
    except Exception as e:
        print(f" {table_name}: ERRO — {e}")
        all_ok = False

print("\n" + "=" * 80)
print("RELATÓRIO DE QUALIDADE DE DADOS")
print("=" * 80 + "\n")

if dq_results:
    df_dq_log = spark.createDataFrame(dq_results)
    
    # Gravar log em Delta
    df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true") \
        .saveAsTable(f"{silver_schema}.dq_log")
    
    # Mostrar resultados
    df_dq_display = df_dq_log.orderBy(F.col("checked_at").desc()).select(
        "table_name", "check_name", "total_rows", "failed_rows", "passed"
    )
    
    display(df_dq_display)
    
    failed_checks = df_dq_log.filter(F.col("passed") == False).count()
    if failed_checks == 0:
        print("\n✓✓✓ TODAS AS VALIDAÇÕES PASSARAM ✓✓✓")
    else:
        print(f"\n  {failed_checks} validações falharam — Revise os blocos acima")
else:
    print("Nenhuma validação registrada")

print("\n" + "=" * 80)
print("✓✓✓ SILVER LAYER PRONTA PARA CONSUMO ✓✓✓")
print("=" * 80)



✓ VALIDAÇÃO FINAL — CAMADA SILVER

Contagem de registros por tabela:

  ✓ tb_info_filmes: 97,879 registros
  ✓ tb_financeiro_filmes: 743,155 registros
  ✓ tb_metricas_engajamento: 751,548 registros
  ✓ tb_avaliacoes_usuarios: 30,761 registros
  ✓ tb_generos: 56,762 registros
  ✓ tb_pessoas_empresas: 328,589 registros
  ✓ tb_cotacao_dolar: 32 registros

RELATÓRIO DE QUALIDADE DE DADOS



table_name,check_name,total_rows,failed_rows,passed
silver.tb_cotacao_dolar,data_cotacao não nula,32,0,true
silver.tb_cotacao_dolar,cotacao_usd_brl > 0,32,0,true
silver.tb_pessoas_empresas,tipo_entidade em domínio válido,328589,0,true
silver.tb_pessoas_empresas,nome_entidade não vazio,328589,0,true
silver.tb_pessoas_empresas,id_filme não nulo,328589,0,true
silver.tb_generos,genero não vazio,56762,0,true
silver.tb_generos,id_filme não nulo,56762,0,true
silver.tb_avaliacoes_usuarios,comentario_usuario nunca vazio,30761,0,true
silver.tb_avaliacoes_usuarios,nome_usuario não nulo,30761,0,true
silver.tb_avaliacoes_usuarios,nota_usuario entre 0 e 10,30761,0,true



  2 validações falharam — Revise os blocos acima

✓✓✓ SILVER LAYER PRONTA PARA CONSUMO ✓✓✓
